# Ingesta de datos - Capa Bronze

Este notebook realiza la ingesta de archivos CSV del Banco Mundial hacia la capa **Bronze** del Data Lake, sin aplicar transformaciones.

## Objetivo

- Leer los archivos fuente con Apache Spark.
- Conservar los datos en estado original (sin limpieza ni enriquecimiento).
- Guardar cada indicador como dataset independiente en `data/bronze`.

In [22]:
from pathlib import Path
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze-worldbank-ingestion")
    .getOrCreate()
)

current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
RAW_ROOT = PROJECT_ROOT / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"

BRONZE_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw root: {RAW_ROOT}")
print(f"Bronze root: {BRONZE_ROOT}")

Project root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation
Raw root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\raw
Bronze root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze


In [ ]:
import csv
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

INDICATOR_FOLDERS = {
    "pib_per_capita": "PIB per cápita (US$ a precios actuales)",
    "indice_gini": "Índice de Gini",
    "desempleo": "Desempleo, total (% de la fuerza laboral total) (estimación modelada de la OIT)",
    "inflacion": "Inflación, deflactor del PIB (%) anual",
    "exportaciones": "Exportaciones de bienes y servicios (% del PIB)",
    "importaciones": "Importaciones de bienes y servicios (% del PIB)",
    "poblacion": "Población, total",
}

CSV_READ_OPTIONS = {
    "header": False,
    "multiLine": False,
    "encoding": "UTF-8",
    "inferSchema": False,
}


def get_world_bank_schema(source_csv_path: Path) -> StructType:
    with open(source_csv_path, "r", encoding="utf-8-sig", newline="") as file:
        non_empty_lines = [line.strip() for line in file if line.strip()]

    header_line = non_empty_lines[2]
    header = next(csv.reader([header_line]))

    return StructType([
        StructField(col_name, StringType(), True)
        for col_name in header
    ])


def read_world_bank_csv(source_csv_path: Path):
    schema = get_world_bank_schema(source_csv_path)

    df = (
        spark.read
        .options(**CSV_READ_OPTIONS)
        .schema(schema)
        .csv(str(source_csv_path))
    )

    first_col = schema.fieldNames()[0]

    return (
        df.filter(F.col(first_col).isNotNull())
        .filter(F.trim(F.col(first_col)) != "")
        .filter(F.col(first_col).isin("Data Source", "Last Updated Date", "Country Name"))
    )

In [ ]:
ingestion_summary = []

for indicator_key, folder_name in INDICATOR_FOLDERS.items():
    source_folder = RAW_ROOT / folder_name

    if not source_folder.exists():
        raise FileNotFoundError(f"No existe la carpeta origen para '{indicator_key}': {source_folder}")

    csv_files = sorted(source_folder.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No se encontró CSV en la carpeta: {source_folder}")

    source_csv = csv_files[0]
    target_path = BRONZE_ROOT / indicator_key

    df_raw = read_world_bank_csv(source_csv)

    (
        df_raw.write
        .mode("overwrite")
        .parquet(str(target_path))
    )

    row_count = df_raw.count()
    year_columns = [c for c in df_raw.columns if c.isdigit() and len(c) == 4]

    ingestion_summary.append(
        {
            "indicador": indicator_key,
            "archivo_origen": source_csv.name,
            "ruta_bronze": str(target_path),
            "registros": row_count,
            "columnas": len(df_raw.columns),
            "anios_detectados": len(year_columns),
            "rango_anios": f"{min(year_columns)}-{max(year_columns)}" if year_columns else "N/A",
        }
    )

print("Ingesta Bronze completada.")
for item in ingestion_summary:
    print(item)
    

Ingesta Bronze completada.
{'indicador': 'pib_per_capita', 'archivo_origen': 'API_NY.GDP.PCAP.CD_DS2_es_csv_v2_2787.csv', 'ruta_bronze': 'c:\\Users\\Darwin Lenis\\OneDrive\\Escritorio\\Mentoria Data\\data-lake-simulation\\data\\bronze\\pib_per_capita', 'registros': 264, 'columnas': 71, 'anios_detectados': 66, 'rango_anios': '1960-2025'}
{'indicador': 'indice_gini', 'archivo_origen': 'API_SI.POV.GINI_DS2_es_csv_v2_5439.csv', 'ruta_bronze': 'c:\\Users\\Darwin Lenis\\OneDrive\\Escritorio\\Mentoria Data\\data-lake-simulation\\data\\bronze\\indice_gini', 'registros': 264, 'columnas': 71, 'anios_detectados': 66, 'rango_anios': '1960-2025'}
{'indicador': 'desempleo', 'archivo_origen': 'API_SL.UEM.TOTL.ZS_DS2_es_csv_v2_1408.csv', 'ruta_bronze': 'c:\\Users\\Darwin Lenis\\OneDrive\\Escritorio\\Mentoria Data\\data-lake-simulation\\data\\bronze\\desempleo', 'registros': 264, 'columnas': 71, 'anios_detectados': 66, 'rango_anios': '1960-2025'}
{'indicador': 'inflacion', 'archivo_origen': 'API_NY.GDP